In [88]:
import pandas as pd
from tools import process_results

qpp_df = pd.read_csv(f'./precomputed_qpps/e5_2_combined_qpp_nq_test.csv')
zero_evals = process_results.load_json(f'../../rag_utility/eval_results/short_answers_0shot_1calls_0_0_bm25_dl_nq_test_concise_eval.json')
k_evals = process_results.load_json(f'../../rag_utility/eval_results/short_answers_2shot_1calls_1_0_e5_dl_nq_test_concise_eval.json')

zero_gens = process_results.load_json(f'../../rag_utility/gen_results/short_answers_0shot_1calls_0_0_bm25_dl_nq_test_concise.json')
k_gens = process_results.load_json(f'../../rag_utility/gen_results/short_answers_2shot_1calls_1_0_e5_dl_nq_test_concise.json')

max_score_df = qpp_df[qpp_df.qpp_method=='maxScore'][['qid', 'query', 'qpp_estimate']]

zero_f1 = [[i[0], i[1]['0']['0']['F1']] for i in zero_evals.items()]
zero_f1 = pd.DataFrame(zero_f1, columns=['qid', 'f1'])

k_f1 = [[i[0], i[1]['0']['0']['F1']] for i in k_evals.items()]
k_f1 = pd.DataFrame(k_f1, columns=['qid', 'f1'])

zero_gens = [[i[0], i[1]['0']['0']['answer']] for i in zero_gens.items() if ('0' in i[1])]
zero_gens = pd.DataFrame(zero_gens, columns=['qid', 'answer'])

k_gens = [[i[0], i[1]['0']['0']['answer']] for i in k_gens.items() if ('0' in i[1].keys())]
k_gens = pd.DataFrame(k_gens, columns=['qid', 'answer'])

zero_f1 = zero_f1.rename(columns={'f1': 'zero_f1'})
zero_gens = zero_gens.rename(columns={'answer': 'zero_answer'})
k_f1 = k_f1.rename(columns={'f1': 'k_f1'})
k_gens = k_gens.rename(columns={'answer': 'k_answer'})

union_f1 = pd.merge(k_f1, zero_f1, on='qid')
union_f1['utility'] = union_f1.k_f1 - union_f1.zero_f1
union_f1 = pd.merge(union_f1, zero_gens, on='qid')
union_f1 = pd.merge(union_f1, k_gens, on='qid')
union_f1 = pd.merge(union_f1, max_score_df, on='qid').sort_values(['qpp_estimate'], ascending=False)


In [169]:
# temp_union_f1 = union_f1.head(200)
negative_utility_qids = union_f1[union_f1.k_f1<0.5].qid.values
positive_utility_qids = union_f1[union_f1.k_f1>=0.5].qid.values
union_f1[union_f1.utility<-0.5].head(50)

,qid,k_f1,zero_f1,utility,zero_answer,k_answer,query,qpp_estimate
2951,test_2951,0.000000,0.750000,-0.750000,\n...per unit volume of blood...,per hundred milliliters,blood alcohol concentration means the parts of...,0.924393
278,test_278,0.000000,1.000000,-1.000000,John Ross,Bo Jackson,who ran the fastest 40 yard dash in the nfl,0.918439
1083,test_1083,0.000000,1.000000,-1.000000,\n...a noble gas,eight electrons,the octet rule states that in chemical compoun...,0.913538
3272,test_3272,0.000000,1.000000,-1.000000,Glenn Close,Betty Lou Gerson,who played cruella de vil in 101 dalmatians,0.913384
2327,test_2327,0.000000,1.000000,-1.000000,Kareem Abdul-Jabbar,Bill Russell,who won the most mvp awards in the nba,0.912014
3054,test_3054,0.000000,0.666667,-0.666667,California's Sierra Nevada mountains,Sutter's Mill Coloma,where did the california gold rush take place,0.911822
3278,test_3278,0.333333,1.000000,-0.666667,Left coronary artery,behind pulmonary artery,where is the left anterior descending artery l...,0.909874
1832,test_1832,0.000000,1.000000,-1.000000,"January 18, 1788",Botany Bay,when did the first fleet arive in australia,0.909325
2560,test_2560,0.000000,0.666667,-0.666667,\nCaucasus,Black Sea-Caspian Steppe,the region that stretches between the black an...,0.907347
1143,test_1143,0.000000,1.000000,-1.000000,"November 22, 1914",mid-November,when did the first battle of ypres end,0.907268


In [161]:
'test_2327'
# union_f1.head(50)

'test_2327'

### Check the document text

In [96]:
import pyterrier as pt

if not pt.java.started():
    pt.java.init()

index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')
text_loader = index.text_loader(["text"])

In [97]:
retr_res = pd.read_csv(f'../../rag_utility/res/e5_nq_test.csv')

In [99]:
retr_res.query("qid=='test_2327'")['query'].values[0]

'who won the most mvp awards in the nba'

In [100]:
text_loader(retr_res[retr_res.qid=='test_2327'].query('rank<2')).text.values

array(['the award a record six times. He is also the only player to win the award despite his team not making the playoffs back in the season. Both Bill Russell and Michael Jordan won the award five times, while Wilt Chamberlain and LeBron James won the award four times. Russell and James are the only players to have won the award four times in five seasons. Moses Malone, Larry Bird and Magic Johnson each won the award three times, while Bob Pettit, Karl Malone, Tim Duncan, Steve Nash and Stephen Curry have each won it twice. Only two rookies have',
       'won the award: Wilt Chamberlain in the and Wes Unseld in the 1968–69 season. Hakeem Olajuwon of Nigeria, Tim Duncan of the U.S. Virgin Islands, Steve Nash of Canada and Dirk Nowitzki of Germany are the only MVP winners considered ""international players"" by the NBA. Stephen Curry in 2015–16 is the only player to have won the award unanimously. Shaquille O\'Neal in 1999–2000 and LeBron James in 2012–13 are the only two players to ha

In [101]:
perpC = process_results.load_json(f'../perplexity_eval/log_prob_temp_res/full_context_with_query/nq_test_e5_2.json')
perpC_indi = process_results.load_json(f'../perplexity_eval/log_prob_temp_res/individual_with_query/nq_test_e5_20.json')

In [182]:
np.median([i[1] for i in perpC.items() if (i[0] in negative_utility_qids)])
for per in [0, 25, 50, 75, 100]:
    print(np.percentile([i[1] for i in perpC.items() if (i[0] in negative_utility_qids)], per))

-3.576171875
-2.462890625
-2.119140625
-1.751708984375
-0.5595703125


In [183]:
np.median([i[1] for i in perpC.items() if (i[0] in positive_utility_qids)])
for per in [0, 25, 50, 75, 100]:
    print(np.percentile([i[1] for i in perpC.items() if (i[0] in positive_utility_qids)], per))

-3.6015625
-2.28466796875
-1.95458984375
-1.590087890625
-0.54248046875


In [135]:
all_perpC = []
negU_perpC = []
for qid, i in perpC_indi.items():
    all_perpC += [i['0'], i['1']]
    if qid in negative_utility_qids:
        negU_perpC += [i['0'], i['1']]

In [139]:
np.median(all_perpC)

-2.19140625

In [138]:
perpC_indi['test_2327']

{'0': -1.69921875,
 '1': -1.9951171875,
 '2': -1.791015625,
 '3': -2.00390625,
 '4': -1.716796875,
 '5': -1.578125,
 '6': -2.1171875,
 '7': -2.5,
 '8': -1.7314453125,
 '9': -1.5205078125,
 '10': -1.244140625,
 '11': -2.59765625,
 '12': -1.9580078125,
 '13': -2.03125,
 '14': -1.978515625,
 '15': -2.162109375,
 '16': -2.01953125,
 '17': -1.9482421875,
 '18': -2.34765625,
 '19': -2.07421875,
 '20': -1.7626953125,
 '21': -2.572265625,
 '22': -1.7099609375,
 '23': -2.849609375}

In [103]:
perpC_indi['test_2327']

-1.650390625

In [111]:
import numpy as np

np.mean(list(perpC.values()))

-2.016945523502424

In [112]:
np.max(list(perpC.values()))

-0.54248046875

In [113]:
np.min(list(perpC.values()))

-3.6015625

In [114]:
np.median(list(perpC.values()))

-2.037109375